In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 13. 임베딩 — 단어를 벡터 공간으로

> **학습 목표**
> - Word2Vec(Skip-gram, CBOW)의 손실 함수를 유도한다
> - GloVe의 동시등장 행렬 접근을 이해한다
> - 트랜스포머의 학습 가능한 토큰 임베딩으로 넘어간다

## 13.1 분산 표현의 철학

"단어의 의미는 그 단어의 동반자에 의해 결정된다" (Firth, 1957).

**분산 표현(distributed representation)**: 의미를 고차원 벡터의 각 차원에 분산시켜 인코딩.
- "king" = [0.9, 0.7, 0.1, ...] (왕족성, 남성성, ...)

## 13.2 Word2Vec — Skip-gram

주어진 단어로 주변 단어를 예측:
$$\max_\theta \sum_{t=1}^{T} \sum_{-c \leq j \leq c, j \neq 0} \log P(w_{t+j} | w_t; \theta)$$

소프트맥스 모델:
$$P(w_O | w_I) = \frac{\exp(\mathbf{v}'_{w_O}{}^\top \mathbf{v}_{w_I})}{\sum_{w \in V} \exp(\mathbf{v}'_w{}^\top \mathbf{v}_{w_I})}$$

- $\mathbf{v}_w$: 단어 $w$의 입력 벡터
- $\mathbf{v}'_w$: 단어 $w$의 출력 벡터
- 분모 계산이 $O(|V|)$로 비싸다 → **Negative Sampling** 사용


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Skip-gram with Negative Sampling (간소화)
class SkipGramNS:
    def __init__(self, vocab_size, embed_dim=50, seed=0):
        rng = np.random.default_rng(seed)
        self.W_in = rng.standard_normal((vocab_size, embed_dim)) * 0.01
        self.W_out = rng.standard_normal((vocab_size, embed_dim)) * 0.01
        self.embed_dim = embed_dim

    def forward(self, center_idx, context_idx, neg_indices):
        v = self.W_in[center_idx]      # (D,)
        u_pos = self.W_out[context_idx]  # (D,)
        u_neg = self.W_out[neg_indices]  # (K, D)
        return v, u_pos, u_neg

    def loss_and_grad(self, center_idx, context_idx, neg_indices):
        v = self.W_in[center_idx]
        u_pos = self.W_out[context_idx]
        u_neg = self.W_out[neg_indices]  # (K, D)

        # positive: log sigma(u_pos . v)
        score_pos = u_pos @ v
        sigma_pos = 1 / (1 + np.exp(-score_pos))
        loss = -np.log(sigma_pos + 1e-12)

        # negative: log sigma(-u_neg . v)
        score_neg = u_neg @ v  # (K,)
        sigma_neg = 1 / (1 + np.exp(-score_neg))
        loss -= np.sum(np.log(1 - sigma_neg + 1e-12))

        # 그래디언트
        dloss_dscore_pos = sigma_pos - 1
        dloss_dscore_neg = sigma_neg  # (K,)

        grad_v = dloss_dscore_pos * u_pos + np.sum(dloss_dscore_neg[:, None] * u_neg, axis=0)
        grad_u_pos = dloss_dscore_pos * v
        grad_u_neg = dloss_dscore_neg[:, None] * v[None, :]

        return loss, grad_v, grad_u_pos, grad_u_neg

    def train_step(self, center_idx, context_idx, neg_indices, lr=0.025):
        loss, gv, gu_pos, gu_neg = self.loss_and_grad(center_idx, context_idx, neg_indices)
        self.W_in[center_idx] -= lr * gv
        self.W_out[context_idx] -= lr * gu_pos
        self.W_out[neg_indices] -= lr * gu_neg
        return loss

# 작은 말뭉치
corpus_text = "the king killed the queen and the prince was sad and the kingdom fell"
words = corpus_text.split()
vocab = sorted(set(words))
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}
vocab_size = len(vocab)
print(f"Vocabulary Size: {vocab_size}, 코퍼스 Length: {len(words)}")

# 학습 데이터 생성 (skip-gram pairs)
window = 2
pairs = []
for i, w in enumerate(words):
    for j in range(max(0, i - window), min(len(words), i + window + 1)):
        if j != i:
            pairs.append((word_to_idx[w], word_to_idx[words[j]]))
print(f"Training 쌍 수: {len(pairs)}")

# 네거티브 샘플링 분포 (unigram^0.75)
word_freqs = Counter(words)
neg_dist = np.array([word_freqs[w]**0.75 for w in vocab])
neg_dist /= neg_dist.sum()

# 학습
model = SkipGramNS(vocab_size, embed_dim=20, seed=0)
losses = []
for epoch in range(50):
    total_loss = 0
    np.random.shuffle(pairs)
    for center, context in pairs:
        K = 5
        negs = np.random.choice(vocab_size, size=K, p=neg_dist)
        total_loss += model.train_step(center, context, negs, lr=0.05)
    losses.append(total_loss / len(pairs))

plt.figure(figsize=(9, 4))
plt.plot(losses, 'b-')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Skip-gram with Negative Sampling Learning Curve')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch13_word2vec_loss.png', dpi=100, bbox_inches='tight')
plt.show()


## 13.3 단어 유추 (Word Analogy)

Word2Vec의 유명한 결과:
$$\mathbf{v}(\text{king}) - \mathbf{v}(\text{man}) + \mathbf{v}(\text{woman}) \approx \mathbf{v}(\text{queen})$$

벡터 공간이 성별, 왕족성 등 의미적 차원을 학습했다.


In [ ]:
# 단어 유사도 계산 (작은 모델)
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)

# 학습된 임베딩으로 유사도
print("Training된 word Embedding Similarity:")
for w1 in ['king', 'queen', 'kingdom']:
    sims = []
    for w2 in vocab:
        if w1 != w2:
            v1 = model.W_in[word_to_idx[w1]]
            v2 = model.W_in[word_to_idx[w2]]
            sims.append((w2, cosine_sim(v1, v2)))
    sims.sort(key=lambda x: -x[1])
    print(f"  '{w1}'와 가장 비슷한 word: {sims[:3]}")


## 13.4 GloVe — 동시등장 통계 기반

GloVe (Pennington et al., 2014)은 단어 쌍의 동시등장(co-occurrence) 통계를 직접 학습.

$$J = \sum_{i,j} f(X_{ij}) \left(\mathbf{w}_i^\top \tilde{\mathbf{w}}_j + b_i + \tilde{b}_j - \log X_{ij}\right)^2$$

- $X_{ij}$: 단어 $i$와 $j$의 동시등장 횟수
- $f(X_{ij})$: 가중 함수 (너무 흔한 단어 쌍 가중치 감소)
- $\mathbf{w}, \tilde{\mathbf{w}}$: 두 개의 단어 벡터 (마지막에 평균)

Word2Vec과 달리 전역 통계를 사용. 큰 코퍼스에서 더 안정적.


In [ ]:
# 동시등장 행렬 계산 (간소화)
def build_cooc_matrix(words, window=2, vocab_size=None, word_to_idx=None):
    """word 동시등장 Matrix."""
    if vocab_size is None:
        vocab = sorted(set(words))
        word_to_idx = {w: i for i, w in enumerate(vocab)}
        vocab_size = len(vocab)
    M = np.zeros((vocab_size, vocab_size))
    for i, w in enumerate(words):
        wi = word_to_idx[w]
        for j in range(max(0, i - window), min(len(words), i + window + 1)):
            if j != i:
                wj = word_to_idx[words[j]]
                # Distance에 따른 Weight
                weight = 1.0 / abs(i - j)
                M[wi, wj] += weight
    return M, word_to_idx

M, w2i = build_cooc_matrix(words, window=2, vocab_size=vocab_size, word_to_idx=word_to_idx)
print(f"동시등장 행렬: {M.shape}")
print(f"상위 5x5:")
print(M[:5, :5].round(2))
print(f"\n'king'과 가장 자주 동시등장하는 word:")
king_idx = word_to_idx['king']
cooc = [(idx_to_word[i], M[king_idx, i]) for i in range(vocab_size) if i != king_idx]
cooc.sort(key=lambda x: -x[1])
for w, c in cooc[:5]:
    print(f"  {w}: {c:.2f}")


## 13.5 트랜스포머의 학습 가능한 토큰 임베딩

Word2Vec/GloVe는 **정적 임베딩** — 같은 단어는 항상 같은 벡터.

문제: "bank" (강둑)과 "bank" (은행)이 같은 벡터 → **다의어 문제**.

트랜스포머는 **학습 가능한 임베딩 행렬** $\mathbf{E} \in \mathbb{R}^{|V| \times d}$:
- 토큰 ID $i$ → $\mathbf{E}[i]$
- 문맥에 따라 어텐션을 통해 동적으로 표현 변경
- "bank"의 임베딩 자체는 고정이지만, 어텐션 후 표현은 문맥 의존

또한 **위치 정보**를 별도로 주입 (Positional Encoding, Ch 16에서 상세).

임베딩 스케일링: 트랜스포머 원 논문은 $\sqrt{d}$를 곱함.
$$\mathbf{E}' = \mathbf{E} \cdot \sqrt{d}$$


In [ ]:
# 트랜스포머 임베딩 (PyTorch)
import torch
import torch.nn as nn

vocab_size = 1000
embed_dim = 64

# 학습 가능한 임베딩
embedding = nn.Embedding(vocab_size, embed_dim)
print(f"Embedding Matrix: {embedding.weight.shape}")
print(f"Parameter Count: {embedding.weight.numel():,}")

# 스케일링
seq = torch.tensor([[1, 5, 10, 25, 100]])  # (batch, seq_len)
emb = embedding(seq)  # (batch, seq_len, embed_dim)
emb_scaled = emb * (embed_dim ** 0.5)

print(f"\nEmbedding Output: {emb.shape}")
print(f"Scaling 전 std: {emb.std():.4f}")
print(f"Scaling 후 std: {emb_scaled.std():.4f} (≈ {embed_dim ** 0.5:.2f}배)")

# 시각화: 임베딩 벡터 (앞 20차원)
fig, ax = plt.subplots(figsize=(12, 4))
emb_sample = embedding.weight[:20, :20].detach().numpy()
im = ax.imshow(emb_sample, cmap='RdBu', aspect='auto')
plt.colorbar(im, ax=ax)
ax.set_xlabel('Embedding Dimension')
ax.set_ylabel('토큰 ID')
ax.set_title('Embedding Matrix (처음 20x20)')
plt.tight_layout()
plt.savefig('../figures/ch13_embedding_matrix.png', dpi=100, bbox_inches='tight')
plt.show()


## 13.6 임베딩 시각화 — t-SNE, PCA

고차원 임베딩을 2D로 축소하여 시각화. 비슷한 단어가 가까이 모임.


In [ ]:
# 임베딩 시각화 (간단한 PCA)
from sklearn.decomposition import PCA

# 학습된 임베딩 가져오기
embeddings = model.W_in  # (vocab_size, embed_dim)
print(f"Embedding: {embeddings.shape}")

# PCA로 2D 축소
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(embeddings)

# 시각화
fig, ax = plt.subplots(figsize=(8, 7))
for i, word in enumerate(vocab):
    ax.scatter(emb_2d[i, 0], emb_2d[i, 1], c='blue', s=80)
    ax.annotate(word, (emb_2d[i, 0], emb_2d[i, 1]), fontsize=11)
ax.set_title('Word2Vec Embedding (PCA 2D 투영)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch13_embedding_tsne.png', dpi=100, bbox_inches='tight')
plt.show()
print("비슷한 word가 가까이 모여 있다 (코퍼스가 작아서 Pattern이 약할 수 있음).")


## 13.7 핵심 정리

| 모델 | 방식 | 특징 |
|---|---|---|
| Word2Vec | 예측 기반 (skip-gram/CBOW) | 빠르고 효율적 |
| GloVe | 동시등장 통계 | 전역 정보 활용 |
| FastText | 서브워드 Word2Vec | OOV 처리 우수 |
| Transformer 임베딩 | 학습 가능 행렬 | 문맥 의존적 표현 |

## 연습문제

1. Skip-gram과 CBOW의 차이를 설명하고, 작은 코퍼스에서 둘 다 구현하라.
2. 음수 샘플링 수 $K = 1, 5, 15$에 따라 학습 결과를 비교하라.
3. GloVe의 가중 함수 $f(x) = (x/x_{\max})^\alpha$ if $x < x_{\max}$ else 1 의 역할을 설명하라.
4. Word2Vec의 단어 유추 실험을 직접 해보고 결과를 분석하라.
5. 트랜스포머 임베딩에 $\sqrt{d}$ 스케일링을 하는 이유를 어텐션 수식과 연관지어 설명하라.

> 해설: `solutions/ch13_solutions.ipynb`
